In [1]:
from yahooquery import Ticker
import pandas as pd
from pathlib import Path

In [2]:
# Top 25 holdings of SPY ETF as of 2026-03-20 (excluding Palantir, which went public in 2020)
stocks = ["NVDA","AAPL","MSFT","AMZN","GOOGL","GOOG","META","AVGO","TSLA",
          "BRK-B","WMT","LLY","JPM","XOM","V","JNJ","MU","MA","COST","ORCL",
          "CVX","NFLX","ABBV","BAC"]

In [3]:
# Scrape Stock data from Yahoo Finance
stock_data = Ticker(stocks)
stock_hist = stock_data.history(start="2015-01-01", end="2025-12-31", interval="1d").reset_index()

In [4]:
# Pull shares outstanding from key_stats
key_stats = stock_data.key_stats
shares_out = (
    pd.DataFrame(key_stats).T
    .reset_index()
    .rename(columns={"index": "symbol"})
    .loc[:, ["symbol", "sharesOutstanding"]]
)

# Merge history with shares outstanding
stock_long = (
    stock_hist
    .merge(shares_out, on="symbol", how="left")
    .loc[:, ["date", "symbol", "open", "high", "low", "adjclose", "volume", "sharesOutstanding"]]
    .dropna(subset=["adjclose"])
)

# Compute approximate market cap and turnover ratio
stock_long["market_cap"] = stock_long["adjclose"] * stock_long["sharesOutstanding"]
stock_long["turnover"] = stock_long["volume"] / stock_long["sharesOutstanding"]

In [5]:
# Pivot to wide format
stock_df = (
    stock_long.assign(date=pd.to_datetime(stock_long["date"]))
    .pivot(
        index="date",
        columns="symbol",
        values=["open", "high", "low", "adjclose", "volume", "turnover", "market_cap"]
    )
    .sort_index()
)

# Flatten MultiIndex columns
stock_df.columns = [f"{field}_{ticker}" for field, ticker in stock_df.columns]

stock_df.head(5)

,open_AAPL,open_ABBV,open_AMZN,open_AVGO,open_BAC,open_BRK-B,open_COST,open_CVX,open_GOOG,open_GOOGL,...,market_cap_META,market_cap_MSFT,market_cap_MU,market_cap_NFLX,market_cap_NVDA,market_cap_ORCL,market_cap_TSLA,market_cap_V,market_cap_WMT,market_cap_XOM
date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,27.8475,65.440002,15.629,10.093,17.99,151.5,141.869995,111.629997,26.378078,26.629999,...,170248072342.021423,295300105876.382568,38160922511.452774,21122451261.67775,11736543944.478035,107805711599.258423,54863060195.253281,102907774731.312286,185113113344.752441,239728233601.62674
2015-01-05,27.0725,65.5,15.3505,10.007,17.790001,148.809998,141.690002,110.959999,26.091366,26.3575,...,167513699672.814636,292584526719.015259,37095706549.373306,20047382649.208469,11538308206.2006,106293863092.918396,52556560736.063309,100636177816.620544,184574344208.29895,233168804657.217102
2015-01-06,26.635,65.620003,15.112,9.895,17.42,147.639999,140.610001,107.870003,25.679497,26.025,...,165256754238.623932,288290111930.052368,36096382580.353241,19704160089.068596,11188486567.139626,105196552100.55542,52854253568.488434,99987721787.769409,185996621130.765289,231929221776.641296
2015-01-07,26.799999,64.57,14.875,9.705,17.139999,147.940002,142.600006,109.25,25.280592,25.547501,...,165256754238.623932,291952958474.544678,35250803732.820419,19806460693.901978,11159333315.491676,105220919235.282898,52771698891.060287,101327356348.529572,190931564555.11322,234279193456.12027
2015-01-08,27.307501,68.160004,15.016,10.053,17.16,150.600006,145.559998,109.190002,24.831326,25.0755,...,169662130573.807983,300541816378.997559,36974909571.154327,20245930401.268616,11579123917.222023,105854958444.213867,52689147792.230148,102686421888.615082,194961392252.862396,238178747053.436432


In [6]:
# find returns
adjclose_cols = [c for c in stock_df.columns if c.startswith("adjclose_")]
price_df = stock_df[adjclose_cols]

stock_ret = price_df.pct_change().dropna(how="all")

/var/folders/lv/z5s79n1j3fs87jp5qd8_r4dm0000gn/T/ipykernel_7584/1308924054.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  stock_ret = price_df.pct_change().dropna(how="all")


In [7]:
# output to csv
DATA_PATH = Path("../Dataset")
stock_df.to_csv(DATA_PATH / "stock_data.csv")
stock_ret.to_csv(DATA_PATH / "stock_returns.csv")